# 08 - Datasets con esquema unificado: real y sintetico

Genera los dos datasets intercambiables para el dashboard de VcM, con **esquema
de columnas identico**:

- **`data/clean/dashboard_real`**: datos 100% reales proyectados sobre el
  esquema unificado. Las columnas que la fuente no captura quedan **presentes
  pero vacias (NaN)**, sin inventar valores. Ademas de alimentar el dashboard,
  ese vacio documenta que datos deberia empezar a capturar VcM.
- **`data/synthetic/dashboard_sintetico`**: mismo esquema, **todas** las columnas
  pobladas, estadisticamente calcado del real (vocabularios reales, proporciones
  reales, distribuciones numericas reales). Criterio guia: **verosimil en
  agregado, ficticio en el detalle**. Solo para demostracion del dashboard.

Insumos: las tablas limpias del notebook 07 (`dashboard_participantes_2024_2025`
y `dashboard_historico_2022_2025`). El real se arma con 2022-2023 desde la tabla
historica y 2024-2025 desde la tabla de participantes, sin duplicar iniciativas.

Documentacion del esquema: `docs/esquema_dashboard.md`.

In [ ]:
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

CLEAN = Path("..") / "data" / "clean"
SYNTH = Path("..") / "data" / "synthetic"
SEED = 42
rng = np.random.default_rng(SEED)

A = pd.read_parquet(CLEAN / "dashboard_participantes_2024_2025.parquet")
B = pd.read_parquet(CLEAN / "dashboard_historico_2022_2025.parquet")
print("Tabla A (participantes 2024-2025):", A.shape)
print("Tabla B (historico 2022-2025):    ", B.shape)

## 1. Esquema unificado

Una lista unica de columnas que cubre todo lo solicitado por la contraparte.
Ambos datasets deben tener exactamente estas columnas, en este orden.

Los flags `sello_*` desagregan `competencia_sello` en las cuatro categorias
pedidas (tecnologico, responsabilidad social, sustentabilidad, genero); se
derivan del texto real normalizado, asi que en el dataset real **si** tienen dato
donde hay sello.

In [ ]:
SCHEMA = [
    # identificacion
    "codigo", "nombre_iniciativa", "facultad", "carrera", "instrumento",
    "anio", "semestre", "estado_sisav",
    # participantes desagregados
    "n_estudiantes", "n_academicos", "n_titulados", "n_empleadores",
    "n_organizaciones_osc",
    # desglose por rol
    "n_titulados_charlista", "n_titulados_expositor", "n_titulados_asistente",
    "n_empleadores_charlista", "n_empleadores_expositor", "n_empleadores_asistente",
    "n_organizaciones_osc_charlista", "n_organizaciones_osc_expositor",
    "n_organizaciones_osc_asistente",
    # competencias
    "competencia_sello", "sello_tecnologia", "sello_responsabilidad_social",
    "sello_sustentabilidad", "sello_genero", "competencia_generica",
    # dominios disciplinares (multivalor; en la fuente real solo desde 2024)
    "dominios_disciplinares",
    # otros indicadores
    "ciclo_estudio", "internacionalizacion", "catedra_asignatura", "comuna_rm",
    # evidencia de ejecucion (SI/NO, normalizada desde la fuente)
    "evidencia",
    # KPI I19
    "cantidad_act_planificadas", "cantidad_act_ejecutadas",
    # linaje
    "_archivo_origen", "_anio", "_origen_dato",
]

DTYPES = {}
for c in SCHEMA:
    if c.startswith("n_") or c.startswith("cantidad_"):
        DTYPES[c] = "float64"
    elif c.startswith("sello_") or c == "internacionalizacion":
        DTYPES[c] = "boolean"
    elif c in ("anio", "_anio"):
        DTYPES[c] = "int64"
    elif c == "ciclo_estudio":
        DTYPES[c] = "Int64"
    else:
        DTYPES[c] = "string"


def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s_]+", " ", s.lower()).strip()


def sello_flags(serie):
    """Deriva los cuatro flags desde el texto de competencia_sello.

    El vocabulario real tiene variantes de mayusculas y forma (TECNOLOGIA /
    Tecnologia, SUSTENTABILIDAD / Sostenibilidad, RESPONSABILIDAD_SOCIAL /
    Responsabilidad social); se comparan normalizadas. NA donde no hay sello.
    """
    flags = {k: [] for k in ("sello_tecnologia", "sello_responsabilidad_social",
                             "sello_sustentabilidad", "sello_genero")}
    for v in serie:
        if pd.isna(v):
            for k in flags:
                flags[k].append(pd.NA)
            continue
        nv = norm(v)
        flags["sello_tecnologia"].append("tecnolog" in nv)
        flags["sello_responsabilidad_social"].append("responsabilidad" in nv)
        flags["sello_sustentabilidad"].append(("sustentab" in nv) or ("sostenib" in nv))
        flags["sello_genero"].append("genero" in nv)
    return {k: pd.array(v, dtype="boolean") for k, v in flags.items()}


def a_esquema(df):
    """Proyecta un DataFrame parcial al esquema unificado con tipos correctos."""
    out = pd.DataFrame(index=df.index)
    for c in SCHEMA:
        if c in df.columns:
            out[c] = df[c].astype(DTYPES[c])
        else:
            vacio = np.nan if DTYPES[c] == "float64" else pd.NA
            out[c] = pd.Series(vacio, index=df.index, dtype=DTYPES[c])
    return out


print(f"Esquema unificado: {len(SCHEMA)} columnas")

## 2. Dataset REAL

Proyeccion de los datos reales sobre el esquema: 2022-2023 desde la tabla
historica, 2024-2025 desde la tabla de participantes (mas rica). Las columnas
sin fuente real (empleadores, roles, semestre, ciclo, internacionalizacion,
comuna) quedan **completamente vacias**: presentes en el esquema, sin inventar
valores. `n_academicos` toma la columna real `n_academicos_docentes`.
`dominios_disciplinares` viene poblada con datos reales donde existen
(2024-2025) y NaN en 2022-2023, coherente con que el dato no existia antes.

Columnas recuperadas de la fuente (datos verdaderos, cobertura desigual por anio):

- `evidencia` (SI/NO): todo el periodo. En 2022-2023 se deriva de `Estado de
  Evidencia` (COMPLETO/INCOMPLETO -> SI, SIN EVIDENCIA -> NO); 2024-2025 traen
  el campo SI/NO directo. Habilita el KPI de % de iniciativas con evidencia.
- `cantidad_act_planificadas`: todo el periodo (existe en la fuente en todos
  los anios).
- `cantidad_act_ejecutadas`: **solo 2022-2023** (la fuente dejo de capturarla
  despues); NaN en 2024-2025. Habilita el KPI I19 real para 2022-2023.

In [ ]:
real_hist = B[B["anio"].isin([2022, 2023])].copy()
real_part = A.copy().rename(columns={"n_academicos_docentes": "n_academicos"})

real = pd.concat([a_esquema(real_hist), a_esquema(real_part)], ignore_index=True)

# flags de sello derivados del dato real (unica desagregacion derivable)
for k, v in sello_flags(real["competencia_sello"]).items():
    real[k] = v

real["_origen_dato"] = pd.Series("real", index=real.index, dtype="string")
real = real[SCHEMA]

print("Dataset real:", real.shape)
print(real["anio"].value_counts().sort_index().to_dict())

In [ ]:
CLEAN.mkdir(parents=True, exist_ok=True)
real.to_csv(CLEAN / "dashboard_real.csv", index=False, encoding="utf-8")
real.to_parquet(CLEAN / "dashboard_real.parquet", index=False)
print("Escrito data/clean/dashboard_real.csv y .parquet")

## 3. Dataset SINTETICO

Mismo esquema exacto, todo poblado, con seed fijo. Como se calca del real:

- **Vocabularios reales**: facultades, carreras, instrumentos, estados,
  competencias, dominios disciplinares y catedras se extraen del dataset real,
  no se inventan nombres.
- **Proporciones reales**: para cada anio se muestrea (bootstrap) el bloque
  categorico completo `(facultad, carrera, instrumento, estado_sisav)` desde las
  filas reales de ese anio. Eso preserva la distribucion conjunta (una carrera
  solo aparece con su facultad real) y el patron temporal (2025 es el anio con
  mas iniciativas, igual que en el real: mismas filas por anio).
- **Distribuciones numericas reales**: los conteos se remuestrean de la
  distribucion empirica real (2024-2025), lo que preserva exactamente min, max,
  media, mediana y desviacion en esperanza; se imprime la comparacion al final.
- **Columnas sin analogo real**: `n_empleadores` se remuestrea de la distribucion
  de organizaciones (orden de magnitud analogo); actividades planificadas 1 a 8
  con probabilidad decreciente.
- **Evidencia**: SI/NO con la **proporcion real observada** de SI (se calcula del
  dataset real y se imprime), no 50/50 arbitrario.
- **Coherencia interna**: los roles (charlista/expositor/asistente) se reparten
  con una multinomial sobre el total de cada categoria, asi **suman exacto**;
  `ejecutadas ~ Binomial(planificadas, 0.85)` nunca supera a planificadas;
  las comunas son comunas reales de la Region Metropolitana.

Lo ficticio en el detalle: codigos `DEMO-...`, nombres genericos de iniciativa,
y la combinacion fila a fila, que no corresponde a ninguna iniciativa real.

In [ ]:
COMUNAS_RM = [
    "Santiago", "Providencia", "Nunoa", "Macul", "San Joaquin", "La Florida",
    "Maipu", "Estacion Central", "Recoleta", "Independencia", "Quinta Normal",
    "Pudahuel", "Cerrillos", "Lo Prado", "Renca", "Quilicura", "Huechuraba",
    "Conchali", "La Cisterna", "San Miguel", "Pedro Aguirre Cerda", "El Bosque",
    "La Granja", "La Pintana", "San Ramon", "Lo Espejo", "Penalolen", "La Reina",
    "Las Condes", "Vitacura", "Lo Barnechea", "Puente Alto", "San Bernardo",
    "Colina", "Melipilla", "Talagante",
]


def resample(values, n):
    """Bootstrap de la distribucion empirica real (sin NaN)."""
    vals = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy()
    return rng.choice(vals, size=n, replace=True)


def repartir_roles(totales, alphas=(1.0, 1.5, 4.5)):
    """Divide cada total en (charlista, expositor, asistente) sumando exacto."""
    out = np.zeros((len(totales), 3))
    for i, t in enumerate(totales):
        t = int(t)
        if t > 0:
            out[i] = rng.multinomial(t, rng.dirichlet(alphas))
    return out


# bloque categorico: bootstrap por anio desde las filas reales de ese anio
bloques = []
for anio, n in real["anio"].value_counts().sort_index().items():
    pool = real.loc[real["anio"] == anio,
                    ["facultad", "carrera", "instrumento", "anio", "estado_sisav"]]
    bloques.append(pool.sample(n=n, replace=True, random_state=SEED + anio))
S = pd.concat(bloques, ignore_index=True)
n = len(S)

# el sintetico va 100% poblado: los NaN heredados del bootstrap se rellenan
# desde el vocabulario real (pares facultad-carrera juntos, para no cruzar
# una carrera con una facultad que no le corresponde)
pares_fc = real[["facultad", "carrera"]].dropna().to_numpy()
faltan = (S["facultad"].isna() | S["carrera"].isna()).to_numpy()
idx = rng.integers(0, len(pares_fc), size=int(faltan.sum()))
S.loc[faltan, ["facultad", "carrera"]] = pares_fc[idx]
estados_reales = real["estado_sisav"].dropna().to_numpy()
faltan_e = S["estado_sisav"].isna().to_numpy()
S.loc[faltan_e, "estado_sisav"] = rng.choice(estados_reales, size=int(faltan_e.sum()))

# identificacion ficticia
S.insert(0, "codigo", [f"DEMO-{i:04d}" for i in range(1, n + 1)])
S.insert(1, "nombre_iniciativa", [f"Iniciativa demo {i:04d}" for i in range(1, n + 1)])
S["semestre"] = rng.choice(["1S", "2S"], size=n)

# conteos calcados de la distribucion empirica real (2024-2025)
S["n_estudiantes"] = resample(A["n_estudiantes"], n)
S["n_academicos"] = resample(A["n_academicos_docentes"], n)
S["n_titulados"] = resample(A["n_titulados"], n)
S["n_organizaciones_osc"] = resample(A["n_organizaciones_osc"], n)
# sin analogo real: empleadores en el orden de magnitud de organizaciones
S["n_empleadores"] = resample(A["n_organizaciones_osc"], n)

# desglose por rol: suma exacta al total de cada categoria
for cat in ("n_titulados", "n_empleadores", "n_organizaciones_osc"):
    roles = repartir_roles(S[cat].to_numpy())
    S[f"{cat}_charlista"] = roles[:, 0]
    S[f"{cat}_expositor"] = roles[:, 1]
    S[f"{cat}_asistente"] = roles[:, 2]

# competencias, dominios y catedra: vocabulario real
S["competencia_sello"] = rng.choice(real["competencia_sello"].dropna().to_numpy(), size=n)
S["competencia_generica"] = rng.choice(A["competencia_generica"].dropna().to_numpy(), size=n)
# dominios: remuestreo de los valores reales (2024-2025), poblado en todos los anios
S["dominios_disciplinares"] = rng.choice(A["dominios_disciplinares"].dropna().to_numpy(), size=n)
S["catedra_asignatura"] = rng.choice(A["catedra_asignatura"].dropna().to_numpy(), size=n)

# otros indicadores
S["ciclo_estudio"] = rng.choice([0, 1, 2, 3], size=n, p=[0.10, 0.30, 0.35, 0.25])
S["internacionalizacion"] = rng.random(n) < 0.08
S["comuna_rm"] = rng.choice(COMUNAS_RM, size=n)

# evidencia SI/NO con la proporcion real observada (no 50/50 arbitrario)
evid_real = real["evidencia"].dropna()
p_si = float((evid_real == "SI").mean())
S["evidencia"] = rng.choice(["SI", "NO"], size=n, p=[p_si, 1 - p_si])
print(f"Proporcion real de evidencia SI: {100 * p_si:.1f}% (usada para el sintetico)")

# KPI I19: ejecutadas nunca supera a planificadas
S["cantidad_act_planificadas"] = rng.choice(
    [1, 2, 3, 4, 5, 6, 7, 8], size=n,
    p=[0.30, 0.22, 0.16, 0.12, 0.08, 0.06, 0.04, 0.02])
S["cantidad_act_ejecutadas"] = rng.binomial(
    S["cantidad_act_planificadas"].astype(int), 0.85)

# linaje
S["_archivo_origen"] = f"generador_sintetico_seed{SEED}"
S["_anio"] = S["anio"]
S["_origen_dato"] = "sintetico"

sintetico = a_esquema(S)
for k, v in sello_flags(sintetico["competencia_sello"]).items():
    sintetico[k] = v
sintetico = sintetico[SCHEMA]

print("Dataset sintetico:", sintetico.shape)
print(sintetico["anio"].value_counts().sort_index().to_dict())

In [ ]:
SYNTH.mkdir(parents=True, exist_ok=True)
sintetico.to_csv(SYNTH / "dashboard_sintetico.csv", index=False, encoding="utf-8")
sintetico.to_parquet(SYNTH / "dashboard_sintetico.parquet", index=False)
print("Escrito data/synthetic/dashboard_sintetico.csv y .parquet")

## 4. Verificacion

Primero las garantias duras (el notebook falla si no se cumplen), despues la
comparacion estadistica real vs sintetico.

In [ ]:
# 1) esquema identico en nombre y orden
assert list(real.columns) == list(sintetico.columns), "Esquemas distintos"

# 2) patron temporal identico al real
assert real["anio"].value_counts().sort_index().to_dict() == \
       sintetico["anio"].value_counts().sort_index().to_dict()

# 3) coherencia interna del sintetico
for cat in ("n_titulados", "n_empleadores", "n_organizaciones_osc"):
    suma = (sintetico[f"{cat}_charlista"] + sintetico[f"{cat}_expositor"]
            + sintetico[f"{cat}_asistente"])
    assert (suma == sintetico[cat]).all(), f"Roles no suman el total en {cat}"
assert (sintetico["cantidad_act_ejecutadas"]
        <= sintetico["cantidad_act_planificadas"]).all()
assert set(sintetico["comuna_rm"].unique()) <= set(COMUNAS_RM)
assert sintetico.notna().all().all(), "El sintetico debe ir 100% poblado"

# 4) el sintetico no filtra identidad real
assert not set(sintetico["nombre_iniciativa"]) & set(real["nombre_iniciativa"].dropna())

print("OK: esquema identico, patron temporal, coherencia interna e identidad.")

In [ ]:
def cobertura(t):
    filas = {}
    for c in t.columns:
        if c in ("anio", "_anio"):
            continue
        s = t[c]
        filas[c] = s.groupby(t["anio"]).apply(
            lambda x: round(100 * x.notna().mean(), 1))
    return pd.DataFrame(filas).T


print("=== COBERTURA dataset REAL (% poblado por anio) ===\n")
print(cobertura(real).to_string())
print("\n=== COBERTURA dataset SINTETICO (% poblado por anio) ===\n")
print(cobertura(sintetico).to_string())

In [ ]:
# confirmacion de las columnas recuperadas de la fuente: cobertura real > 0%
FOCO = ["evidencia", "cantidad_act_planificadas", "cantidad_act_ejecutadas"]
print("=== COBERTURA de las columnas recuperadas (% poblado por anio) ===\n")
for nombre, t in (("REAL", real), ("SINTETICO", sintetico)):
    tab = pd.DataFrame({c: t.groupby("anio")[c].apply(
        lambda s: round(100 * s.notna().mean(), 1)) for c in FOCO}).T
    print(f"-- {nombre} --")
    print(tab.to_string(), "\n")

assert (real["evidencia"].notna().mean() > 0), "evidencia vacia en el real"
assert (real["cantidad_act_planificadas"].notna().mean() > 0), "planificadas vacia en el real"
assert (real["cantidad_act_ejecutadas"].notna().mean() > 0), "ejecutadas vacia en el real"
print("OK: las tres columnas tienen cobertura real > 0%.")

In [ ]:
# comparacion de distribuciones clave
stats = ["mean", "std", "50%", "min", "max"]
comp = {}
pares = [("n_estudiantes", "n_estudiantes"), ("n_academicos", "n_academicos"),
         ("n_titulados", "n_titulados"), ("n_organizaciones_osc", "n_organizaciones_osc")]
for cr, cs in pares:
    dr = real[cr].dropna().describe()
    ds = sintetico[cs].describe()
    comp[f"{cr} (real)"] = dr[stats]
    comp[f"{cr} (sint)"] = ds[stats]
print("=== Distribuciones de conteos: real (2024-2025) vs sintetico ===\n")
print(pd.DataFrame(comp).T.round(1).to_string())

prop = pd.DataFrame({
    "real": real["facultad"].value_counts(normalize=True).round(3),
    "sintetico": sintetico["facultad"].value_counts(normalize=True).round(3),
}).fillna(0)
print("\n=== Proporcion de iniciativas por facultad ===\n")
print(prop.to_string())

propi = pd.DataFrame({
    "real": real["instrumento"].value_counts(normalize=True).round(3),
    "sintetico": sintetico["instrumento"].value_counts(normalize=True).round(3),
}).fillna(0)
print("\n=== Proporcion por instrumento ===\n")
print(propi.to_string())

## 5. Notas de uso

- **Esquema y semantica de cada columna**: `docs/esquema_dashboard.md`.
- El dataset **sintetico es SOLO para demostracion del dashboard**: sus cifras no
  corresponden a ninguna iniciativa real y no sirven para reporteria, informes
  ni acreditacion.
- Las columnas vacias del dataset **real** son la hoja de ruta de captura para
  VcM: cada NaN estructural marca un dato que el instrumento deberia empezar a
  registrar (empleadores, roles, internacionalizacion, comuna, semestre, ciclo,
  actividades ejecutadas).
- Real en `data/clean/` (confidencial, gitignoreado); sintetico en
  `data/synthetic/` (ficticio, versionable). Nunca mezclados.